# CA6011 Deep Learning for NLP: Week 6 Lab -- Transformers


In this lab, we'll learn to build a GPT-Style (Decoder-only) transformer model from scratch. we'll cover key components like self-attention/causal attention, positional encoding, and decoder layer. We'll train this model on a language modelling task using a Wikipedia dataset. For detailed information, refer to the Week 6 lecture slides and [Speech and Language Processing (3rd ed. draft, Feb 2024) Dan Jurafsky and James H. Martin, Chapter 10](https://web.stanford.edu/~jurafsky/slp3/old_feb24/10.pdf).



# 0  Preliminaries

Let's start by setting up imports and GPU usage.

In [ ]:
# Import the clear_output function from IPython.display module for clearing Jupyter Notebook cell output
from IPython.display import clear_output
# Install the necessary libraries for handling datasets and tokenization
# Tokenizers library from Hugging Face, specifically for creating and training custom tokenizers
!pip install datasets tokenizers
clear_output()

In [ ]:
# Import the main PyTorch library
import torch

# Import the torch.nn module, which contains neural network-related classes and functions
import torch.nn as nn
import torch.nn.functional as F


# Import the torch.optim module, which provides optimization algorithms for updating neural network parameters
import torch.optim as optim

# Import the DataLoader class from torch.utils.data module, used for loading datasets in mini-batches during training
from torch.utils.data import Dataset, DataLoader

# imports to train custom tokenizer
from tokenizers import Tokenizer  # Main class for tokenization processes
from tokenizers.models import BPE  # BPE model for Byte Pair Encoding tokenization
from tokenizers.trainers import BpeTrainer  # Trainer class for training BPE tokenizers
from tokenizers.pre_tokenizers import Whitespace  # Pre-tokenizer that splits on whitespace

# To load datasets from Huggingface Data Hub
from datasets import load_dataset

# Import the Iterable and List classes/types from typing module for type hinting
from typing import Iterable, List

# Import tqdm to track for loop progress
from tqdm import tqdm

# Import the NumPy library with the alias np for numerical operations
import numpy as np

# Import the random module for generating random numbers
import random

# Import the math module for mathematical functions and constants
import math

# Import the time module for working with time
import time

Let's set the random seeds for deterministic results.

In [ ]:
SEED = 1234
# Setting a seed ensures reproducibility of results in random processes.
# By setting the seed to a fixed value, random number generation becomes deterministic,
# meaning that the same sequence of random numbers will be generated each time the code is run.

# Set the seed for Python's built-in random module
random.seed(SEED)

# Set the seed for NumPy's random number generator
np.random.seed(SEED)

# Set the seed for PyTorch's random number generator on CPU
torch.manual_seed(SEED)

# Set the seed for PyTorch's random number generator on GPU (if available)
torch.cuda.manual_seed(SEED)

# Ensure deterministic behavior for cuDNN (CUDA Deep Neural Network library) operations in PyTorch
# This helps in ensuring reproducibility when using CUDA (GPU acceleration) for deep learning operations
torch.backends.cudnn.deterministic = True


It's time to use some GPUs, to do that, we need to define a `torch.device`. This is used to tell pytorch to put all the tensors in our code on the GPU or not. We use the `torch.cuda.is_available()` function, which will return `True` if a GPU is detected on our computer. We pass this `device` to the iterator.

In [ ]:
# Check if CUDA (GPU acceleration) is available on the system
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

To display GPU device details you are using, run the following command line.

**RUN ONLY WHEN CUDA IS AVAILABLE**

In [ ]:
# Execute the shell command to display GPU information using nvidia-smi
!nvidia-smi

# 1 Transformers (Decoder-only-GPT Style) Implementation

## 1.1 Attention Block Implementation

![](https://drive.google.com/uc?export=view&id=1OI33bLv1MBy7QXMK2zKV3pTnLYvFuxur)

The above diagram shows a multihead attention layer with four attention heads, followed by a downproject back down to *d*, the dimensionality of the model.

The inputs in **X** are multiplied by the query, key and value weight matrices respectively, to yield the query, key and value vectors:

$\textbf{Q} = \textbf{XW}^Q; \textbf{K} = \textbf{XW}^K; \textbf{V} = \textbf{XW}^V$

$\textbf{A} = SelfAttention(\textbf{Q}, \textbf{K}, \textbf{V}) = softmax(\frac{\textbf{QK}^T}{\sqrt d_k})\textbf{V}$

$softmax(\frac{\textbf{QK}^T}{\sqrt d_k})$ is the **Attention score**.

Where:

- $\textbf{X}$: The input matrix representing the sequence of tokens.
- $\textbf{W}^Q, \textbf{W}^K,  \textbf{W}^V$: Weight matrices for queries, keys, and values, respectively. These matrices are learned parameters that transform the input X into the query ($\textbf{Q}$), key ($\textbf{K}$), and value ($\textbf{V}$) representations.
- A: is a contexualised vector embedding representation for each token in the input influenced by the attention scores.

Here, we will implement **causal attention**, which is the mechanism used in transformer models, like GPT (Generative Pre-trained Transformer), to ensure that a token can only attend to preceding tokens in the sequence, not future ones. This is crucial for tasks like text generation, where the model predicts the next token based on the previous context. By masking future tokens (as indicated in the diagram below), causal attention mimics the forward-only nature of reading and writing, allowing the model to generate coherent and contextually relevant text based on the input it has seen so far. For more details, see the lecture slides and Jurafsky & Martin, [Chapter 10](https://web.stanford.edu/~jurafsky/slp3/old_feb24/10.pdf).

![](https://drive.google.com/uc?export=view&id=1Manc-woaDxQoFavzM3OwbK6_e0bUZ2KV)


**NOTE**: For matrix reshaping, we use `.view()`. The `.view()` method in PyTorch is used for reshaping a tensor without copying its data. It returns a new tensor with the same data as the original tensor but with a specified new shape. In contrast, `reshape()` creates a new copy of the data. Moreover, `.view` is more memory-efficient, an advantage especially significant when working with large matrices.

For `.view()` to work, the new shape must be compatible with the original shape, and the tensor must be contiguous in memory. If the tensor is not contiguous, you might need to call `.contiguous()` before using `.view()`. The `.contiguous()` method ensures that a tensor is stored in a contiguous block of memory. After operations like `transpose()`, `permute()`, or slicing, which may result in a tensor that does not have its data stored contiguously, `.contiguous()` can be used to make the tensor contiguous. It rearranges the tensor data in memory, if necessary.


In [ ]:
class MultiHeadCausalAttention(nn.Module):
    """
    Implements a multi-head causal attention mechanism for autoregressive models.

    This module divides the input embeddings into multiple heads, applies scaled dot-product attention
    with a causality mask to each head independently, and then projects the results
    back to the original embedding dimension.
    """
    def __init__(self, embed_size, heads):
        """
          Parameters:
            - embed_size (int): The size of the input and output embeddings.
            - heads (int): The number of attention heads, which allows the model to jointly attend to
                information from different representation subspaces at different positions.

          Attributes:
            - values (nn.Linear): Linear transformation for the values.
            - keys (nn.Linear): Linear transformation for the keys.
            - queries (nn.Linear): Linear transformation for the queries.
            - proj_out (nn.Linear): Output projection layer to transform concatenated head outputs
                back to the original embedding size.
        """
        super(MultiHeadCausalAttention, self).__init__()
        self.embed_size = embed_size  # Dimension of the input embeddings
        self.heads = heads  # Number of attention heads
        # Ensure the embedding size is divisible by the number of heads for equal division
        assert embed_size % heads == 0, "Embedding size needs to be divisible by heads"

        # Define linear transformations for queries, keys, and values
        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        # Output projection layer
        self.proj_out = nn.Linear(embed_size, embed_size) # Linear layer to project concatenated outputs back to embed size

    def forward(self, x):
        """
        Forward pass for the multi-head causal attention layer.

        Parameters:
        - x (Tensor): Input tensor of shape [batch_size, seq_len, embed_size].

        Returns:
        - Tensor: Output tensor of shape [batch_size, seq_len, embed_size], after applying
          multi-head causal attention.
        """
        # Extract dimensions: batch size, sequence length, and embedding size
        batch_size, seq_len, embed_size = x.size()  # x: [batch_size, seq_len, embed_size]

        # Process queries, keys, values: split into 'heads' number of heads, each with head dimension (embed_size/heads)
        # Using .view we divide queries, keys and values for each head
        ## Before transpose: [batch_size, seq_len, heads, embed_size/heads] | After transpose: [batch_size, heads, seq_len, embed_size/heads]
        all_queries = self.queries(x).view(batch_size, seq_len, self.heads, embed_size//self.heads).transpose(1,2)  # [batch_size, heads, seq_len, embed_size/heads]
        all_keys = self.keys(x).view(batch_size, seq_len, self.heads, embed_size//self.heads).transpose(1,2)      # Same as above for keys
        all_values = self.values(x).view(batch_size, seq_len, self.heads, embed_size//self.heads).transpose(1,2)  # Same as above for values

        # Compute scaled dot-product attention
        queries_keys = (all_queries @ all_keys.transpose(-1,-2)) * (1/math.sqrt(all_keys.size(-1)))  # Scale by sqrt of key dimension

        # Create a causal mask to mask out future tokens (prevent attending to future positions)
        causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1).to(queries_keys.device)  # Upper triangular matrix with '-inf' above the diagonal

        # Apply the causal mask by adding it to the scaled dot-product scores
        masked_queries_keys = queries_keys + causal_mask[None, None, :, :]  # Add causal mask, broadcasting it to match dimensions

        # Apply softmax to get attention probabilities, ensuring masked positions are near zero (after softmax)
        attn_score = F.softmax(masked_queries_keys, dim=-1)  # [batch_size, heads, seq_len, seq_len]

        # Multiply attention scores by values and sum to get weighted sum representation
        out = (attn_score @ all_values).transpose(1,2).contiguous().view(batch_size, seq_len, embed_size)  # [batch_size, seq_len, embed_size]

        # Apply the final linear projection layer
        out = self.proj_out(out)  # Transform back to original embed size

        return out

In [ ]:
# Example usage
embed_size = 256
heads = 8
seq_length = 10
attn_model = MultiHeadCausalAttention(embed_size, heads)

# Dummy input tensor simulating a batch of sequences
x = torch.rand((1, seq_length, embed_size))  # [batch_size, seq_length, embed_size]

out = attn_model(x)
print(out.shape)
# Expected output shape: (1, seq_length, embed_size), demonstrating the model maintains input dimensions


## 1.2 Position-wise Feed-Forward Block Implementation

Let's look at how to implement the feed-forward layer. This consists of two linear transformations with a ReLU activation in between, and is applied independently to each position. See lecture slides and Jurafsky & Martin, chapter 10 for details.

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, embed_size, ff_size):
        """
        Implements a position-wise feed-forward network as used in Transformer models.

        This network consists of two linear transformations with a ReLU activation in between,
        applied independently to each position.

        Parameters:
            - embed_size (int): The size of the input and output embeddings. In the context of
                Transformers, this is typically the size of the model's hidden states.
            - ff_size (int): The size of the hidden layer in the feed-forward network. This is often
                larger than the embedding size (4x in most cases),
                allowing the network to project the inputs into a higher
                dimensional space before applying the second linear transformation and ReLU activation.

        Attributes:
          - fc1 (nn.Linear): The first linear layer that projects input embeddings from `embed_size`
            to `ff_size`.
          - fc2 (nn.Linear): The second linear layer that projects the output of the ReLU activation
            back down to `embed_size`.
        """
        super().__init__()
        # First linear transformation (embedding size to feed-forward size)
        self.fc1 = nn.Linear(embed_size, ff_size)
        # Second linear transformation (feed-forward size back to embedding size)
        self.fc2 = nn.Linear(ff_size, embed_size)

    def forward(self, x):
        """
        Defines the forward pass of the position-wise feed-forward network.

        Parameters:
        - x (Tensor): The input tensor with shape [batch size, sequence length, embed_size].

        Returns:
        - Tensor: The output tensor with the same shape as the input, after applying two
          linear transformations with a ReLU activation in between.
        """
        ## INSERT YOUR CODE HERE ##
        # Apply the first linear layer followed by a ReLU activation
        x = F.relu(self.fc1(x))
        # Apply the second linear layer
        x = self.fc2(x)
        ## END OF YOUR CODE ##
        return x

## 1.3 Causal Attention Block + Point-wise Feed-Forward Block = Decoder Layer

![](https://drive.google.com/uc?export=view&id=1UbLDEvWZr20kSbmTiYdkhhhmGW8Un6uh)

We will also apply layer normalisation *before* each block (note the diagram shows post-norm, but see the lecture slides for pre-norm), and employ residual connections after each block. This is motivated by several practical and theoretical considerations that enhance the model's training stability and performance. You can refer to these resources to understand more about the role of layer normalisation and residual connections in training stability:
- [Transformers without Tears: Improving the Normalization of Self-Attention](https://aclanthology.org/2019.iwslt-1.17/)
- [On Layer Normalization in the Transformer Architecture](https://arxiv.org/abs/2002.04745)



In [ ]:
class GPTDecoderLayer(nn.Module):
    """
    Represents a single layer of a GPT-style decoder, incorporating a causally masked
    multi-head attention mechanism and a position-wise feed-forward network.

    Each layer processes the input sequence through two main blocks:
    1. A causally masked multi-head self-attention mechanism that ensures each position
       can only attend to earlier positions in the sequence, preserving the autoregressive property.
    2. A position-wise feed-forward network that applies two linear transformations with a ReLU
       activation in between, independently to each position.

    Layer normalisation is applied before each block, and residual connections are employed
    after each block, following the "pre-norm" architecture where normalisation is applied
    before the block operations.

  """

    def __init__(self, embed_size, heads, ff_size):
        """
        Parameters:
          - embed_size (int): The size of the input and output embeddings, representing the dimensionality
            of the model's hidden states.
          - heads (int): The number of heads in the multi-head attention mechanism. Multiple heads allow
            the model to capture information from different representation subspaces.
          - ff_size (int): The size of the hidden layer in the position-wise feed-forward network. This is
            typically larger than the embedding size (4x in most cases).

        Attributes:
          - norm1 (nn.LayerNorm): The layer normalisation applied before the multi-head attention mechanism.
          - norm2 (nn.LayerNorm): The layer normalisation applied before the position-wise feed-forward network.
          - attention (MultiHeadCausalAttention): The causally masked multi-head attention module.
          - feed_forward (PositionwiseFeedForward): The position-wise feed-forward network module.
        """
        super().__init__()
        self.norm_attn = nn.LayerNorm(embed_size)  # Pre-attention layer normalisation
        self.norm_ff = nn.LayerNorm(embed_size)  # Pre-feed-forward layer normalisation
        self.attention = MultiHeadCausalAttention(embed_size, heads)  # Causal attention mechanism
        self.feed_forward = PositionwiseFeedForward(embed_size, ff_size)  # Feed-forward network

    def forward(self, x):
        """
        Defines the forward pass of the GPT decoder layer.

        Parameters:
        - x (Tensor): The input tensor of shape [batch_size, sequence_length, embed_size].

        Returns:
        - Tensor: The output tensor of the same shape as the input, after processing through
          the causally masked multi-head attention and the position-wise feed-forward network.
        """
        ## INSERT YOUR CODE HERE ##
        # Apply causally masked multi-head self-attention
        attn = self.attention(self.norm_attn(x))  # Normalise before attention
        x = x + attn  # Add residual connection (residual stream)

        # Apply position-wise feed-forward network
        ff = self.feed_forward(self.norm_ff(x))  # Normalise before feed-forward
        x = x + ff  # Add residual connection (residual stream)
        ## END OF YOUR CODE ##
        return x

In [ ]:
# Example usage
embed_size = 512  # Size of the token embeddings
heads = 8         # Number of attention heads
ff_size = 2048    # Size of the inner feed-forward layers

# Creating a single GPT-style decoder layer
decoder_layer = GPTDecoderLayer(embed_size, heads, ff_size)

## 1.4 Positional Encoding

Positional encoding adds information about the position of each token in the sequence to the embeddings. Since transformers do not inherently process data sequentially like RNNs, they lack a mechanism to recognise the order of tokens. Positional encodings solve this by providing a unique identifier for each position, enabling the model to learn about position-dependent patterns.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, seq_len=5000):
        super(PositionalEncoding, self).__init__()
        # Initialise a positional encoding matrix with zeros
        pe = torch.zeros(seq_len, embed_size)
        # Create a position tensor from 0 to seq_len
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        # Compute the division term for scaling the positional encoding
        div_term = torch.exp(torch.arange(0, embed_size, 2).float() * (-math.log(10000.0) / embed_size))
        # Apply sine to even indices in the positional encoding matrix
        pe[:, 0::2] = torch.sin(position * div_term)
        # Apply cosine to odd indices in the positional encoding matrix
        pe[:, 1::2] = torch.cos(position * div_term)
        # Add a batch dimension and transpose the matrix
        pe = pe.unsqueeze(0)
        # Register pe as a buffer that is not a model parameter, but should be part of the model
        # and move with model between devices.
        self.register_buffer('pe', pe)
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

## 1.5 GPT Model Implementation

In the implementation below we are using Stacked Decoder layers as in the GPT Model family.

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, embed_size, ff_size, num_layers, num_heads):
        """
        Initialise the GPT model with necessary components.

        Parameters:
        - vocab_size (int): The size of the vocabulary. Determines the number of rows in the embedding layer.
        - embed_size (int): The size of each embedding vector. Determines the number of columns in the embedding layer.
        - num_layers (int): The number of decoder layers to include in the model.
        - num_heads (int): The number of heads in the multi-head attention mechanism within each GPTDecoderLayer.
        """
        super(GPTModel, self).__init__()  # Initialise the parent class (nn.Module)
        self.embedding = nn.Embedding(vocab_size, embed_size)  # Create an embedding layer
        self.positional_encoding = PositionalEncoding(embed_size)
        # Create a ModuleList of GPTDecoderLayer instances. ModuleList is a PyTorch container for holding submodules.
        self.layers = nn.ModuleList([GPTDecoderLayer(embed_size, num_heads, ff_size) for _ in range(num_layers)])

        self.un_embedding = nn.Linear(embed_size, vocab_size)

    def forward(self, input_ids):
        """
        Define the forward pass of the GPT model.

        Parameters:
        - input_ids (Tensor): A tensor containing a batch of input token IDs.

        Returns:
        - Tensor: The output from the last layer of the model.
        """
        embeddings = self.embedding(input_ids)  # Convert input token IDs to embeddings
        x = self.positional_encoding(embeddings)  # Apply positional encoding to the embeddings
        for layer in self.layers:  # Iterate over each GPTDecoderLayer in the model
            x = layer(x) # Pass the input (or output from the previous layer) through the current layer
        logits = self.un_embedding(x)
        return logits  # Return the output from the last layer


# 2 Prepare Dataset

## 2.1 Tokenization

**[Byte Pair Encoding (BPE)](https://arxiv.org/abs/1508.07909)** is a data compression technique that has been adapted for use in natural language processing, specifically for tokenizing text. It works by iteratively merging the most frequently occurring pair of characters or character sequences in a given text dataset. Initially, every character is treated as a separate token, and BPE finds and merges the most common adjacent pairs of tokens, creating new, longer tokens in the process. This merging process continues for a specified number of iterations or until a desired vocabulary size is reached, leading to a set of tokens that includes commonly used words, subwords, and characters. This approach helps in handling rare or out-of-vocabulary words by breaking them down into understandable subword units, making it particularly useful for language models in handling a wide range of vocabularies efficiently.

For a detailed description of how it works, refer to this [huggingface blogpost](https://huggingface.co/learn/nlp-course/en/chapter6/5)

Now, we will load our dataset from huggingface. We will use the Wikitext corpus to train our model. Wikitext consists of over 100 million tokens extracted from verified Good and Featured articles on Wikipedia. The dataset is large, and retains original punctuation, numbers, and case, unlike many other datasets. It comes in several versions, including WikiText-2 and WikiText-103, offering a range of sizes suitable for different computational capacities. You can find the dataset and further details on the [Hugging Face Datasets page](https://huggingface.co/datasets/wikitext).

In [ ]:
# Load the WikiText-2 dataset, specifically the raw version for the training split
# This dataset is used for natural language processing tasks like language modeling
# Load the training split
wiki_train_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split='train')

# Load the validation split
wiki_valid_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split='validation')

# Load the test split
wiki_test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split='test')

In [ ]:
wiki_train_dataset

In [ ]:
wiki_train_dataset['text'][4]

Let's train our BPE Tokenizer.

In [ ]:
# Initialise tokenizer and trainer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

#Creates a trainer with a vocabulary size of 5000,
#a minimum frequency threshold of 5 for tokens to be included in the vocabulary,
# and specifies special token
trainer = BpeTrainer(
    vocab_size=10_000,
    min_frequency=5,
    special_tokens=["[UNK]", "[SEP]"]
)

#Sets a pre-tokenizer to split the texts on whitespace, preparing them for BPE tokenization.
tokenizer.pre_tokenizer = Whitespace()

#Trains the tokenizer on the texts dataset using the defined trainer.
tokenizer.train_from_iterator(wiki_train_dataset['text'], trainer=trainer)

# Save the tokenizer
tokenizer.save("./wiki-text-bpe.tokenizer.json")

# Check Vocab Size:
print(f"Vocab Size: {tokenizer.get_vocab_size() }")

## 2.2 Dataset Class

Now, we need to create a Torch container for our dataset so that we can input it into our model.

In [ ]:
class LanguageModelingDataset(Dataset):
    def __init__(self, texts, tokenizer, seq_length):
        """
        Args:
            texts (list of str): The list of sentences.
            tokenizer: The tokenizer instance (trained BPE tokenizer).
            seq_length (int): The maximum length of the tokenized output.
        """
        self.tokenizer = tokenizer
        self.seq_length = seq_length

        # Tokenize texts and prepare input-target pairs
        self.input_ids = []
        for text in texts:
            encoded_text = tokenizer.encode(text).ids
            # Ensure the sequence is within seq_length
            encoded_text = encoded_text[:seq_length]
            self.input_ids.extend(encoded_text)

    def __len__(self):
        # The dataset length is now the total number of tokens minus seq_length,
        # because we need to create input-target pairs
        return len(self.input_ids) - seq_length

    def __getitem__(self, idx):
        """
        Generates one sample of data, which is a pair of adjacent tokens.
        """
        chunk = self.input_ids[idx:idx+self.seq_length]
        # Input sequence is the current token
        input_id = chunk[:-1]
        # Target sequence is the next token
        target_id = chunk[1: ]

        # Convert to tensors
        input_id = torch.tensor(input_id, dtype=torch.long)
        target_id = torch.tensor(target_id, dtype=torch.long)

        return input_id, target_id

In [ ]:
# Instantiate our datasets inside torch datast container:
# Load your pretrained BPE tokenizer
tokenizer = Tokenizer.from_file("./wiki-text-bpe.tokenizer.json")
seq_length = 512  # Maximum sequence length


# Instantiate the dataset
# due to our compute budget, we will train/eval/test in small subset of the data
train_dataset = LanguageModelingDataset(wiki_train_dataset['text'][:800], tokenizer, seq_length)

valid_dataset = LanguageModelingDataset(wiki_valid_dataset['text'][:500], tokenizer, seq_length)

test_dataset = LanguageModelingDataset(wiki_test_dataset['text'][:500], tokenizer, seq_length)

# 3 Training

## 3.1 Training Utils

Let's start with hyperparameters, initialisation and other basic aspects of training setup.

In [ ]:
# Example usage
vocab_size = tokenizer.get_vocab_size()  # Assume this gets the vocab size from your tokenizer
embed_size = 512  # Example embedding size
ff_size = 1024 # Size of the inner feed-forward layers
num_layers = 6  # Example number of layers
num_heads = 8  # Example number of attention heads
BATCH_SIZE = 8 #Batch Size

model = GPTModel(vocab_size, embed_size, ff_size, num_layers, num_heads).to(device)

Next up is initialising the weights of our model. In the paper they state they initialise all weights from a normal distribution with mean 0 and standard deviation 0.2 .

We initialise weights in PyTorch by creating a function which we `apply` to our model. When using `apply`, the `init_weights` function will be called on every module and sub-module within our model. For each module we loop through all of the parameters and sample them from a uniform distribution with `nn.init.uniform_`.

In [ ]:
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.normal_(param.data, mean=0.0, std=0.02)

model.apply(init_weights)

We also define a function that will calculate the number of trainable parameters in the model.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

We will use an Adam optimiser, same as the Week 5 Lab.

In [ ]:
learning_rate = 1e-5
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

Next, we define our loss function. The `CrossEntropyLoss` function calculates both the log softmax as well as the negative log-likelihood of our predictions.


In [ ]:
criterion = nn.CrossEntropyLoss()

## 3.2 Training Loop

Let's first define a train function.

In [ ]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE)

    for input, target in tqdm(dataloader, total=len(dataloader), desc='Training'):
        input = input.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(input)

        #target = [target len, batch size]
        #output = [target len, batch size, output dim]

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        target = target[1:].view(-1)

        #target = [(target len - 1) * batch size]
        #output = [(target len - 1) * batch size, output dim]
        criterion = criterion.to(device)
        loss = criterion(output, target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(list(dataloader))

Now, we need an evaluate function, before we can start training.

In [ ]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    dataloader = DataLoader(iterator, batch_size=BATCH_SIZE)
    with torch.no_grad():
        for input, target in tqdm(dataloader, total=len(dataloader), desc='Eval'):

            input = input.to(device)
            target = target.to(device)


            output = model(input)

            #target = [target len, batch size]
            #output = [target len, batch size, output dim]

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            target = target[1:].view(-1)

            #target = [(target len - 1) * batch size]
            #output = [(target len - 1) * batch size, output dim]

            loss = criterion(output, target)


            epoch_loss += loss.item()

        return epoch_loss / len(list(dataloader))

Next, we'll create a function that we'll use to tell us how long an epoch takes.

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
N_EPOCHS = 10
CLIP = 1

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_dataset, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_dataset, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'GPT-LM-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):.3f}')

After training our model and saving it at best validation loss, we can load the parameters (`state_dict`) and you can start using it for inference.

In [ ]:
model.load_state_dict(torch.load('GPT-LM-model.pt'))

loss = evaluate(model, test_dataset, criterion)

print(f'| Loss: {loss:.3f} | PPL: {math.exp(loss):.3f} |')

# Week 6 Submission Task

In this task, you will update the provided codebase to implement an encoder-only **BERT** (Bidirectional Encoder Representations from Transformers) style Transformer, using our GPT-style decoder-only model as a starting point.

---

## What's the difference between GPT and BERT?

| GPT (Current Lab)                | BERT (Your Task)                |
|-----------------------------------|---------------------------------|
| Causal Attention (only looks back) | Full Self-Attention (looks both ways) |
| Language Modeling (next token prediction) | Masked Language Modeling (MLM) or Next Sentence Prediction (NSP) |
| Decoder-only architecture        | Encoder-only architecture       |

---

**Hints :)**

1. **Replace Causal Attention with Full Self-Attention**: you can use the same codebase; you just need to change the causal attention (looking backward) to full self-attention (looking both backward and forward).

2. **Change the Dataset Class**: in a BERT-style model we perform masked-token or next-sentence predition, while in GPT-style models we perform next-token prediction. In masked-token prediction, we randomly mask around 15% of the tokens, replacing the original tokens with a [MASK] token ID. The target is to predict the original tokens at those masked positions. Apply this change to other affected parts of the code accordingly.

For (1), you just need to delete two lines. For (2), [here](https://github.com/coaxsoft/pytorch_bert/blob/master/bert/dataset.py) is a good starting point.

Also, check out Section 3 in the [BERT paper](https://arxiv.org/abs/1810.04805). Section 3.1 of the BERT paper describes two tasks used for pretraining; just implement one of them (either the masked LM (MLM) or the next sentence prediction (NSP)).

**NOTE:** What we are doing here in this lab is pretraining. Our implementation and comments explain how the pretraining phase is conducted.

# Resources

- Speech and Language Processing (3rd ed. draft) Dan Jurafsky and James H. Martin
  - [Chapter 10](https://web.stanford.edu/~jurafsky/slp3/old_feb24/10.pdf), Sections 10.1-10.6.

